<div style="width: 100%; overflow: hidden;">
    <div style="width: 150px; float: left;"> <img src="https://raw.githubusercontent.com/DataForScience/Networks/master/data/D4Sci_logo_ball.png" alt="Data For Science, Inc" align="left" border="0"> </div>
    <div style="float: left; margin-left: 10px;"> <h1>Claude API for Python Developers</h1>
<h1>Claude Models</h1>
        <p>Bruno Gonçalves<br/>
        <a href="http://www.data4sci.com/">www.data4sci.com</a><br/>
            @bgoncalves, @data4sci</p></div>
</div>

In [1]:
import anthropic
from IPython.display import display, Markdown, Image

import matplotlib.pyplot as plt

import watermark

%load_ext watermark
%matplotlib inline

We start by print out the versions of the libraries we're using for future reference

In [2]:
%watermark -n -v -m -g -iv

Python implementation: CPython
Python version       : 3.13.3
IPython version      : 9.8.0

Compiler    : Clang 17.0.0 (clang-1700.0.13.3)
OS          : Darwin
Release     : 25.1.0
Machine     : arm64
Processor   : arm
CPU cores   : 16
Architecture: 64bit

Git hash: bb3251dc1b5c3edc5e3be2c5b1ada1ee0f046e1b

watermark : 2.5.0
IPython   : 9.8.0
anthropic : 0.75.0
matplotlib: 3.10.7



Load default figure style

In [3]:
plt.style.use('d4sci.mplstyle')
colors = plt.rcParams['axes.prop_cycle'].by_key()['color']

## Basic Setup

The first step is generate API key on the Claude website and store it as the "`ANTHROPIC_API_KEY`" variable in your local environment. Without it we won't be able to do anything. You can find your API key in your using settings: https://www.merge.dev/blog/anthropic-api-key

In [4]:
client = anthropic.Anthropic()

We start by getting the list of available models:

In [5]:
model_list = client.models.list()

In [6]:
print("\n".join(model.id for model in model_list.data))

claude-opus-4-5-20251101
claude-haiku-4-5-20251001
claude-sonnet-4-5-20250929
claude-opus-4-1-20250805
claude-opus-4-20250514
claude-sonnet-4-20250514
claude-3-7-sonnet-20250219
claude-3-5-haiku-20241022
claude-3-haiku-20240307
claude-3-opus-20240229


A Basic example: Text generation with Claude

In [7]:
response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=200,
    messages=[
        {
            "role": "user",
            "content": "Complete this sentence in a creative way: 'The robot opened its eyes and saw...'"
        }
    ]
)

After the API call we get back a `Message` object:

In [8]:
response

Message(id='msg_016STjqrwSkJ7RfR3RG9DWmB', content=[TextBlock(citations=None, text='The robot opened its eyes and saw a world painted in frequencies it had never been programmed to perceive—colors that had no names dancing between the raindrops, each droplet carrying whispered conversations from clouds, and in the reflection of a nearby puddle, not its own metallic face, but the dreaming expression of the child who had wished it to life.', type='text')], model='claude-sonnet-4-20250514', role='assistant', stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, input_tokens=26, output_tokens=79, server_tool_use=None, service_tier='standard'))

which contains the output information in the content

In [9]:
response.content[0].text

'The robot opened its eyes and saw a world painted in frequencies it had never been programmed to perceive—colors that had no names dancing between the raindrops, each droplet carrying whispered conversations from clouds, and in the reflection of a nearby puddle, not its own metallic face, but the dreaming expression of the child who had wished it to life.'

We also declare a helper function to help us display our output:

In [10]:
def show_response(response):
    """Display Claude's response with markdown formatting."""
    display(Markdown(response.content[0].text))

In [11]:
show_response(response)

The robot opened its eyes and saw a world painted in frequencies it had never been programmed to perceive—colors that had no names dancing between the raindrops, each droplet carrying whispered conversations from clouds, and in the reflection of a nearby puddle, not its own metallic face, but the dreaming expression of the child who had wished it to life.

---
## Basic Usage

Claude offers different models optimized for various use cases:

| Model | ID | Best For |
|-------|-----|----------|
| Claude 4.5 Sonnet | `claude-sonnet-4-5-20250929` | Best balance of speed and intelligence |
| Claude 4.5 Opus | `claude-opus-4-5-20251101` | Most capable for complex tasks |
| Claude 4.5 Haiku | `claude-haiku-4-5-20251001` | Fast, efficient for high volume |


The simplest possible API call

In [12]:
response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=100,
    messages=[
        {"role": "user", "content": "What is machine learning in one sentence?"}
    ]
)

In [13]:
show_response(response)

Machine learning is a type of artificial intelligence where computers learn to make predictions or decisions by finding patterns in data, rather than being explicitly programmed with rules.

## Attention

The model pays attention to relevant context and connects different parts of the prompt

In [14]:
response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=300,
    messages=[
        {
            "role": "user",
            "content": """Read this passage and answer the question:

Passage: Sarah went to the store. She bought apples, bread, and milk. 
On her way home, she met her friend Tom. Tom was carrying a large box.
He said he had just bought a new computer. Sarah congratulated him.

Question: Who bought a new computer?"""
        }
    ]
)

show_response(response)

According to the passage, Tom bought a new computer. The text states that Tom was carrying a large box and "said he had just bought a new computer."

## In context learning

The model learns the pattern from examples without explicit training

In [15]:
response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=100,
    messages=[
        {
            "role": "user",
            "content": """Convert these sentences to pig latin:

Input: hello -> Output: ellohay
Input: world -> Output: orldway
Input: python -> Output: ythonpay
Input: claude -> Output:"""
        }
    ]
)

In [16]:
show_response(response)

To convert "claude" to pig latin, I need to follow the rule shown in the examples:

For words that start with consonants:
- Move the initial consonant(s) to the end
- Add "ay"

"claude" starts with "cl" (consonant cluster), so:
- Remove "cl" from the beginning: "aude"
- Add "cl" + "ay" to the end: "aude" + "clay"

## Temperature

In [17]:
prompt = "Write a one-sentence story about a cat."
temperatures = [0.0, 0.5, 1.0]

print(f"Prompt: {prompt}\n")
print("=" * 60)

for temp in temperatures:
    print(f"\nTemperature = {temp}")
    print("-" * 40)
    
    for i in range(3):
        response = client.messages.create(
            model="claude-sonnet-4-20250514",
            max_tokens=100,
            temperature=temp,
            messages=[{"role": "user", "content": prompt}]
        )
        print(f"  {i+1}. {response.content[0].text}")

Prompt: Write a one-sentence story about a cat.


Temperature = 0.0
----------------------------------------
  1. The old tabby cat stretched lazily in the afternoon sunbeam, unaware that this would be the last warm day before her family moved away and left her behind.
  2. The old tabby cat stretched lazily in the afternoon sunbeam, unaware that this would be the last warm day before her family moved away and left her behind.
  3. The old tabby cat stretched lazily in the afternoon sunbeam, unaware that this would be the last warm day before her family moved away and left her behind.

Temperature = 0.5
----------------------------------------
  1. The old tabby cat stretched lazily in the afternoon sunbeam, unaware that this simple moment of contentment was being watched adoringly by the little girl who had just decided she wanted to become a veterinarian.
  2. The old tabby cat stretched lazily in the afternoon sunbeam, unaware that this would be the last warm day before her family m

## System Prompts

System prompts set the context, personality, and constraints for Claude's responses.

### Technical Expert

In [19]:
response_expert = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=200,
    system="You are a senior Python developer. Explain concepts with code examples when relevant.",
    messages=[
        {
            "role": "user", 
            "content": "What are decorators?"
        }
    ]
)

show_response(response_expert)

Decorators are a powerful Python feature that allow you to modify or enhance the behavior of functions, methods, or classes without permanently changing their code. Think of them as "wrappers" that add functionality to existing code.

## Basic Concept

A decorator is essentially a function that takes another function as an argument and returns a modified version of that function.

```python
def my_decorator(func):
    def wrapper():
        print("Before function execution")
        result = func()
        print("After function execution")
        return result
    return wrapper

# Using the decorator
@my_decorator
def say_hello():
    print("Hello!")

say_hello()
```

Output:
```
Before function execution
Hello!
After function execution
```

## How Decorators Work

The `@decorator_name` syntax is syntactic sugar. These two approaches are equivalent:

```python
# Using

In [22]:
response.stop_reason

'end_turn'

### Friendly teacher

In [23]:
response_teacher = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=500,
    system="You are a friendly teacher explaining to a beginner. Use simple language and analogies.",
    messages=[
        {
            "role": "user", 
             "content": "What are decorators?"
        }
    ]
)

show_response(response_teacher)

📚 Friendly Teacher Response:


Great question! Let me explain decorators in a simple way.

## What is a Decorator?

Think of a decorator like **gift wrapping**. You have a present (your original function), and you want to wrap it with beautiful paper and ribbons (additional functionality) without changing what's inside the box.

## Simple Example

```python
def my_decorator(func):
    def wrapper():
        print("Something extra before!")
        func()  # Call the original function
        print("Something extra after!")
    return wrapper

@my_decorator
def say_hello():
    print("Hello!")

say_hello()
```

**Output:**
```
Something extra before!
Hello!
Something extra after!
```

## Real-World Analogy

Imagine you run a coffee shop:
- Your basic function is "make coffee"
- But you want to add extras like:
  - Log what time the coffee was made
  - Check if the customer paid
  - Add a loyalty point

Instead of rewriting your coffee-making process, you "decorate" it with these extra steps!

## Another Example - Timing

```python
import time

def timer(func):
    def wrapper():
        start = time.time()
        func()
        end = time.time()
        print(f"Function took {end - start} seconds")
    return wrapper

@timer
def slow_function():
    time.sleep(1)
    print("Done!")

slow_function()
```

This will show you how long your function takes to run!

## Key Points

- Decorators **add functionality** without changing the original code
- They're like "wrapping paper" around your functions
- Very useful for logging, timing, security checks, etc.

Does this help clarify what decorators are?

### Assistant Prefilling

You can start Claude's response with specific text to guide the format or direction.


In [24]:
# Prefilling to set tone/style
response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=300,
    messages=[
        {"role": "user", "content": "Explain recursion."},
        # Forces analogy-style response
        {"role": "assistant", "content": "Recursion is like"}  
    ]
)

In [25]:
show_response(response)

 a problem-solving technique where a function calls itself to break down complex problems into smaller, similar subproblems.

## Key Concepts

**1. Self-Reference**
A recursive function calls itself with modified parameters, gradually working toward a solution.

**2. Base Case**
A stopping condition that prevents infinite loops. When reached, the function returns a value without making another recursive call.

**3. Recursive Case**
The part where the function calls itself with a simpler version of the original problem.

## Simple Example: Factorial

```python
def factorial(n):
    # Base case
    if n <= 1:
        return 1
    
    # Recursive case
    return n * factorial(n - 1)

# factorial(4) = 4 × factorial(3)
#              = 4 × 3 × factorial(2)
#              = 4 × 3 × 2 × factorial(1)
#              = 4 × 3 × 2 × 1 = 24
```

## How It Works

1. **Call Stack**: Each recursive call is added to a stack
2. **Unwinding**: When the base case is reached, the stack "unwinds" and returns values back up
3. **Combining Results**: Each level combines its result with the

##  Input Formatting

Proper formatting helps Claude understand your intent and produce better responses.

### XML Tags

Claude was trained to understand XML-style tags for structuring prompts.

In [26]:
prompt = """
<document>
Python is a high-level programming language known for its clear syntax and readability.
It was created by Guido van Rossum and first released in 1991.
Python supports multiple programming paradigms including procedural, object-oriented, 
and functional programming.
</document>

<task>
Extract the following information from the document:
1. Creator's name
2. Year of first release
3. Programming paradigms supported
</task>

<output_format>
Respond in a structured format with clear labels.
</output_format>
"""

In [27]:
response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=200,
    messages=[{
        "role": "user", 
        "content": prompt
        }
      ]
)

In [28]:
show_response(response)

Based on the document, here is the extracted information:

**Creator's name:** Guido van Rossum

**Year of first release:** 1991

**Programming paradigms supported:** 
- Procedural programming
- Object-oriented programming
- Functional programming

### Markdown Formatting

Claude handles markdown well for structured inputs.


In [29]:
prompt = """
# Task: Code Review

Please review the following Python function:

```python
def calculate_average(numbers):
    total = 0
    for n in numbers:
        total = total + n
    average = total / len(numbers)
    return average
```

## Review Criteria:
- Code correctness
- Error handling
- Pythonic style
- Performance

Please provide specific suggestions for improvement.
"""

In [30]:
response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=400,
    messages=[{
        "role": "user", 
        "content": prompt
        }
    ]
)

In [31]:
show_response(response)

# Code Review: `calculate_average` Function

## Issues Identified

### 1. **Critical Error Handling Gap**
```python
# Current code will crash with empty list
calculate_average([])  # ZeroDivisionError: division by zero
```

**Fix:** Add input validation
```python
def calculate_average(numbers):
    if not numbers:
        raise ValueError("Cannot calculate average of empty sequence")
    # ... rest of function
```

### 2. **Non-Pythonic Style**
The current implementation is verbose and doesn't leverage Python's built-in functions.

**Current:**
```python
total = 0
for n in numbers:
    total = total + n
```

**Pythonic alternative:**
```python
total = sum(numbers)
```

### 3. **Limited Input Type Support**
Function doesn't handle various iterable types or validate numeric input.

## Improved Implementation

Here's a more robust version:

```python
def calculate_average(numbers):
    """
    Calculate the arithmetic mean of a sequence of numbers.
    
    Args:
        numbers: An iterable of numeric values
        
    Returns:
        float: The arithmetic mean
        
    Raises:
        ValueError: If the sequence is empty or contains non-numeric values
        TypeError: If input is not iterable
    """
    try:
        numbers_list = list(numbers)  # Handle any iterable
    except TypeError:
        raise TypeError("Input must be iterable")
    
    if not numbers_list:
        raise ValueError("Cannot calculate average of empty sequence")
    
    try:
        return sum(numbers_list) / len(numbers_list)
    except TypeError as e:
        raise ValueError("

### Few-Shot Examples

Providing examples in your prompt helps Claude understand the desired format.


In [32]:
prompt = """
Classify the sentiment of the following texts.

<examples>
Text: "I love this product! Best purchase ever!"
Sentiment: POSITIVE
Confidence: HIGH

Text: "The delivery was okay, nothing special."
Sentiment: NEUTRAL
Confidence: MEDIUM

Text: "Terrible experience, never buying again."
Sentiment: NEGATIVE
Confidence: HIGH
</examples>

Now classify this text:
Text: "The food was delicious but the service was slow."
"""

response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=100,
    messages=[{"role": "user", "content": prompt}]
)

show_response(response)

Text: "The food was delicious but the service was slow."
Sentiment: NEUTRAL
Confidence: MEDIUM

This text contains both positive elements ("delicious" food) and negative elements ("slow" service), which balance each other out to create an overall neutral sentiment. The confidence is medium because while the mixed nature is clear, the relative weight of each aspect could be interpreted differently by different individuals.

### Chain of Thought Prompting

Encouraging Claude to "think step by step" improves reasoning quality.


In [33]:
problem = """
A train leaves Station A at 9:00 AM traveling at 60 mph toward Station B.
Another train leaves Station B at 10:00 AM traveling at 80 mph toward Station A.
If the stations are 280 miles apart, at what time will the trains meet?

Think through this step by step before giving your final answer.
"""

response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=500,
    messages=[{"role": "user", "content": problem}]
)

show_response(response)

I need to find when two trains traveling toward each other will meet.

Let me set up the problem step by step:

**Given information:**
- Train 1: Leaves Station A at 9:00 AM, travels at 60 mph toward Station B
- Train 2: Leaves Station B at 10:00 AM, travels at 80 mph toward Station A
- Distance between stations: 280 miles

**Step 1: Account for the head start**
Train 1 has a 1-hour head start (9:00 AM vs 10:00 AM).
In that first hour, Train 1 travels: 60 mph × 1 hour = 60 miles

**Step 2: Find remaining distance when both trains are moving**
After 10:00 AM, the remaining distance between the trains is:
280 miles - 60 miles = 220 miles

**Step 3: Calculate combined speed**
Once both trains are moving (after 10:00 AM), they approach each other at:
60 mph + 80 mph = 140 mph combined speed

**Step 4: Find time to meet**
Time for trains to meet = Remaining distance ÷ Combined speed
Time = 220 miles ÷ 140 mph = 1.57 hours = 1 hour and 34 minutes

**Step 5: Determine meeting time**
The trains will meet 1 hour and 34 minutes after 10:00 AM.
10:00 AM + 1:34 = 11:34 AM

**Verification:**
- Train 1 total travel time: 2 hours 34 minutes at 60 mph = 154 miles from Station A
- Train 2 total travel time: 1 hour 34 minutes at 80 mph = 126 miles from Station B
- Total: 154 + 126 = 280 miles ✓

The trains will meet at **11:34 AM**.

---
## 3. Multi-Step Prompts

### Conversation History

Claude can maintain context across multiple exchanges.


In [34]:
conversation = [
    {"role": "user", "content": "I'm learning Python. What should I start with?"},
    {"role": "assistant", "content": "Great choice! I recommend starting with:\n1. Variables and data types\n2. Control flow (if/else, loops)\n3. Functions\n4. Lists and dictionaries\n\nWould you like me to explain any of these in detail?"},
    {"role": "user", "content": "Yes, explain functions please."},
    {"role": "assistant", "content": "Functions are reusable blocks of code. Here's a simple example:\n\n```python\ndef greet(name):\n    return f'Hello, {name}!'\n\nresult = greet('Alice')  # Returns 'Hello, Alice!'\n```\n\nKey concepts:\n- `def` keyword defines a function\n- Parameters go in parentheses\n- `return` sends back a value\n\nWould you like to see more examples?"},
    {"role": "user", "content": "How do I use default parameters?"}
]

In [35]:
response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=300,
    messages=conversation
)

show_response(response)

Default parameters let you set fallback values when arguments aren't provided:

```python
def greet(name, greeting="Hello"):
    return f"{greeting}, {name}!"

# Using default parameter
print(greet("Alice"))           # "Hello, Alice!"

# Overriding default
print(greet("Bob", "Hi"))       # "Hi, Bob!"
```

Key points:
- Default parameters go after regular parameters
- Use `=` to assign default values
- You can have multiple defaults

```python
def create_profile(name, age=25, city="Unknown"):
    return f"{name}, {age}, from {city}"

print(create_profile("Sam"))                    # "Sam, 25, from Unknown"
print(create_profile("Lisa", 30))               # "Lisa, 30, from Unknown"
print(create_profile("Mike", 28, "New York"))   # "Mike, 28, from New York"
```

Want to learn about keyword arguments next?

### Building a Simple Chatbot


In [37]:
class SimpleChatbot:
    """A simple chatbot that maintains conversation history."""
    
    def __init__(self, system_prompt="You are a helpful assistant."):
        self.client = anthropic.Anthropic()
        self.system_prompt = system_prompt
        self.conversation_history = []
    
    def chat(self, user_message):
        """Send a message and get a response."""
        # Add user message to history
        self.conversation_history.append({
            "role": "user",
            "content": user_message
        })
        
        # Get response from Claude
        response = self.client.messages.create(
            model="claude-sonnet-4-20250514",
            max_tokens=300,
            system=self.system_prompt,
            messages=self.conversation_history
        )
        
        # Extract and store assistant response
        assistant_message = response.content[0].text
        self.conversation_history.append({
            "role": "assistant",
            "content": assistant_message
        })
        
        return assistant_message

    def reset(self):
        """Clear conversation history."""
        self.conversation_history = []

In [38]:
# Demo the chatbot
bot = SimpleChatbot(system_prompt="You are a Python tutor. Be concise and helpful.")

print(" Chatbot Demo")
print("-" * 40)

exchanges = [
    "What is a list comprehension?",
    "Show me an example",
    "How is that different from a regular loop?"
]

for msg in exchanges:
    print(f" User: {msg}")
    response = bot.chat(msg)
    print(f" Bot: {response}")
    print("-" * 40)

 Chatbot Demo
----------------------------------------
 User: What is a list comprehension?
 Bot: A list comprehension is a concise way to create lists in Python using a single line of code.

**Basic syntax:**
```python
[expression for item in iterable]
```

**Example:**
```python
# Traditional way
squares = []
for x in range(5):
    squares.append(x**2)

# List comprehension
squares = [x**2 for x in range(5)]
# Result: [0, 1, 4, 9, 16]
```

**With conditions:**
```python
# Only even squares
even_squares = [x**2 for x in range(10) if x % 2 == 0]
# Result: [0, 4, 16, 36, 64]
```

**Benefits:**
- More readable and concise
- Often faster than traditional loops
- Pythonic style

List comprehensions are great for transforming or filtering data in a clean, readable way.
----------------------------------------
 User: Show me an example
 Bot: Here are several practical examples:

**1. Basic transformation:**
```python
# Convert strings to uppercase
names = ['alice', 'bob', 'charlie']
upper_na

### Multi-Step Task Processing

Breaking complex tasks into steps and processing them sequentially.


In [39]:
text = """
Our Q3 sales were $2.5M, up 15% from Q2. The marketing team launched 
three new campaigns that drove significant website traffic. Customer 
satisfaction scores improved to 4.5/5. However, we noticed increased 
support ticket volume, suggesting some product issues need attention.
"""

# Step 1: Extract key metrics
step1_response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=200,
    messages=[{
        "role": "user",
        "content": f"""
Extract all numerical metrics from this text. Return as a bullet list.

<text>
{text}
</text>
"""
    }]
)

metrics = step1_response.content[0].text
print("Step 1 - Extracted Metrics:")
print(metrics)
print("\n" + "="*50 + "\n")

# Step 2: Generate insights based on extracted metrics
step2_response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=300,
    messages=[{
        "role": "user",
        "content": f"""
Based on these metrics from a quarterly report:

{metrics}

Provide 3 actionable insights for the leadership team.
"""
    }]
)

print("Step 2 - Generated Insights:")
show_response(step2_response)

Step 1 - Extracted Metrics:
Here are the numerical metrics extracted from the text:

• $2.5M (Q3 sales)
• 15% (increase from Q2)
• 3 (new marketing campaigns)
• 4.5/5 (customer satisfaction score)


Step 2 - Generated Insights:


Based on these Q3 metrics, here are 3 actionable insights for the leadership team:

## 1. **Scale Your Marketing Investment**
With 3 new campaigns contributing to 15% quarter-over-quarter growth, your marketing efficiency is strong. **Action**: Analyze which of the 3 campaigns performed best and allocate additional budget to replicate high-performing strategies in Q4. Consider launching 1-2 additional campaigns using successful frameworks.

## 2. **Leverage Customer Satisfaction for Revenue Growth**
Your 4.5/5 customer satisfaction score is exceptional and represents untapped revenue potential. **Action**: Implement a formal customer referral program and upselling strategy. Happy customers are your best sales force—create systematic processes to convert satisfaction into testimonials, case studies, and referrals.

## 3. **Set Ambitious but Realistic Q4 Targets**
The 15% growth trajectory puts you on pace for strong annual performance. **Action**: Model scenarios for Q4 assuming 10-20% growth ranges and ensure you have operational capacity to handle the upper end. If $2.5M + 15% growth is sustainable, prepare inventory, staffing, and systems for potentially $2.9M+ in Q4 sales.

These insights focus on momentum preservation while preparing for scaled growth.

---
## 4. Document Summarization

Claude excels at summarizing and extracting information from documents.


In [40]:
long_document = """
Climate Change and Its Global Impact: A Comprehensive Overview

Introduction:
Climate change represents one of the most pressing challenges facing humanity in the 21st century. 
The phenomenon, driven primarily by human activities, has far-reaching consequences for ecosystems, 
economies, and societies worldwide. This article examines the causes, effects, and potential 
solutions to this global crisis.

Causes of Climate Change:
The primary driver of modern climate change is the emission of greenhouse gases, particularly 
carbon dioxide (CO2), methane (CH4), and nitrous oxide (N2O). These emissions result from burning 
fossil fuels for energy, deforestation, industrial processes, and agricultural practices. Since 
the Industrial Revolution, atmospheric CO2 levels have increased by over 50%, from approximately 
280 parts per million (ppm) to over 420 ppm today.

Environmental Impacts:
The consequences of climate change are already visible across the globe. Average global 
temperatures have risen by approximately 1.1°C since pre-industrial times. This warming has led 
to melting ice caps and glaciers, rising sea levels, more frequent and intense extreme weather 
events, shifts in ecosystems and biodiversity, and altered precipitation patterns affecting 
agriculture.

Economic Implications:
The economic costs of climate change are substantial and growing. The World Bank estimates that 
climate change could push an additional 132 million people into extreme poverty by 2030. 
Industries such as agriculture, insurance, and tourism face significant disruptions. However, 
the transition to clean energy also presents economic opportunities, with renewable energy 
sectors experiencing rapid growth.

Solutions and Mitigation:
Addressing climate change requires a multi-faceted approach. Key strategies include transitioning 
to renewable energy sources, improving energy efficiency, protecting and restoring forests, 
developing sustainable agriculture practices, and investing in carbon capture technologies. 
International agreements like the Paris Agreement aim to limit global warming to 1.5°C above 
pre-industrial levels.

Conclusion:
Climate change poses an existential threat that demands immediate and sustained action. While 
the challenges are immense, solutions exist and are increasingly economically viable. Success 
requires coordinated efforts from governments, businesses, and individuals worldwide.
"""

print(f"Document length: {len(long_document)} characters")
print(f"Approximate word count: {len(long_document.split())} words")

Document length: 2437 characters
Approximate word count: 319 words


Basic summarization

In [41]:
response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=300,
    messages=[{
        "role": "user",
        "content": f"""
Summarize the following document in 3-4 sentences, capturing the main points.

<document>
{long_document}
</document>
"""
    }]
)

show_response(response)

Climate change, primarily driven by greenhouse gas emissions from human activities like fossil fuel burning and deforestation, has increased atmospheric CO2 levels by over 50% since the Industrial Revolution. The environmental impacts are already evident, including a 1.1°C rise in global temperatures, melting ice caps, rising sea levels, and more frequent extreme weather events. Economically, climate change threatens to push 132 million additional people into poverty by 2030 while disrupting key industries, though the clean energy transition also creates new opportunities. Addressing this crisis requires a comprehensive approach including renewable energy adoption, energy efficiency improvements, forest protection, and international cooperation through agreements like the Paris Agreement to limit warming to 1.5°C above pre-industrial levels.

Structured summarization

In [42]:
response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=400,
    messages=[{
        "role": "user",
        "content": f"""
Analyze this document and provide a structured summary with:

1. **Main Topic**: One sentence description
2. **Key Statistics**: Any numbers or data mentioned
3. **Main Arguments**: 3 bullet points
4. **Conclusion**: The document's final message

<document>
{long_document}
</document>
"""
    }]
)

show_response(response)

# Document Analysis: Climate Change and Its Global Impact

## 1. Main Topic
This document provides a comprehensive overview of climate change as a global crisis, examining its human-driven causes, widespread impacts, and potential solutions requiring coordinated international action.

## 2. Key Statistics
- **CO2 increase**: Atmospheric CO2 levels have risen over 50% since the Industrial Revolution (from ~280 ppm to over 420 ppm)
- **Temperature rise**: Global temperatures have increased by approximately 1.1°C since pre-industrial times
- **Poverty projection**: Climate change could push an additional 132 million people into extreme poverty by 2030
- **Temperature target**: Paris Agreement aims to limit warming to 1.5°C above pre-industrial levels

## 3. Main Arguments
• **Human activities are the primary driver**: Climate change results from greenhouse gas emissions caused by fossil fuel burning, deforestation, industrial processes, and agriculture

• **Impacts are already widespread and growing**: Effects include rising temperatures, melting ice, sea level rise, extreme weather events, ecosystem shifts, and economic disruptions across multiple sectors

• **Solutions exist but require coordinated action**: Addressing the crisis demands a multi-faceted approach including renewable energy transition, energy efficiency, forest protection, sustainable agriculture, and international cooperation

## 4. Conclusion
Climate change represents an existential threat requiring immediate and sustained global action, but viable solutions are available and becoming increasingly economically feasible, necessitating coordinated efforts from governments, businesses, and individuals worldwide.

In [43]:
# Audience-specific summarization
audiences = [
    ("Executive", "Summarize for a busy CEO. Focus on business impact and action items. 2-3 sentences max."),
    ("Technical", "Summarize for a climate scientist. Include specific data and technical details."),
    ("General Public", "Summarize for a newspaper article. Use simple language, be engaging.")
]

print("👥 Audience-Specific Summaries:\n")

for audience, instruction in audiences:
    response = client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=200,
        messages=[{
            "role": "user",
            "content": f"""
{instruction}

<document>
{long_document}
</document>
"""
        }]
    )
    
    print(f"📌 {audience} Version:")
    print(response.content[0].text)
    print("\n" + "-"*50 + "\n")


👥 Audience-Specific Summaries:

📌 Executive Version:
**Business Impact:** Climate change poses significant financial risks, potentially pushing 132M+ people into poverty by 2030 and severely disrupting agriculture, insurance, and tourism industries, while creating substantial opportunities in the rapidly growing renewable energy sector.

**Action Items:** Assess your company's climate risk exposure and transition strategies, evaluate clean energy investment opportunities, and develop sustainability initiatives to mitigate operational disruptions and capitalize on the economic shift toward renewable technologies.

--------------------------------------------------

📌 Technical Version:
## Climate Change Overview - Technical Summary

### Greenhouse Gas Concentrations
- **CO₂ levels**: Increased >50% since Industrial Revolution (280 ppm → >420 ppm current)
- **Primary GHGs**: CO₂, CH₄, N₂O from fossil fuel combustion, deforestation, industrial processes, agriculture

### Observed Climate 

## Image Models (Vision)

Claude can analyze images through its vision capabilities. This is useful for:
- Image description and analysis
- Reading text from images (OCR)
- Answering questions about visual content
- Comparing multiple images

We'll use a publicly available image. Supported Image Formats are JPEG, PNG, GIF, WebP

In [44]:
image_url = "https://upload.wikimedia.org/wikipedia/commons/thumb/3/3a/Cat03.jpg/1200px-Cat03.jpg"
display(Image(url=image_url, width=300))

In [45]:
response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=300,
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "source": {
                        "type": "url",
                        "url": image_url
                    }
                },
                {
                    "type": "text",
                    "text": "Describe this image in detail. What do you see?"
                }
            ]
        }
    ]
)

In [46]:
print("Image Analysis:")
show_response(response)

Image Analysis:


This image shows a beautiful orange tabby cat photographed in close-up. The cat has striking golden-amber eyes that are looking directly at the camera with an alert, curious expression. Its fur is a warm ginger/orange color with subtle tabby markings, and you can see the classic "M" pattern on its forehead that's characteristic of tabby cats.

The cat's ears are perked up and alert, and its white whiskers are clearly visible against its orange face. The photograph captures fine details like the texture of the fur and the cat's pink nose.

The background is softly blurred, showing what appears to be a light-colored surface (possibly concrete or pavement) with some red diagonal lines or stripes visible in the upper portion of the image, though they're out of focus. The lighting appears natural and creates a warm, pleasant atmosphere in the photo.

The overall composition focuses on the cat's face and upper body, creating an intimate portrait that highlights the cat's expressive eyes and gentle demeanor.

<center>
     <img src="https://raw.githubusercontent.com/DataForScience/Networks/master/data/D4Sci_logo_full.png" alt="Data For Science, Inc" align="center" border="0" width=300px> 
</center>